# Maintenance

## 共通処理

In [ ]:
import os

from assistant_agent.store import PostgresStoreConnector

engine = PostgresStoreConnector(os.environ["AA_PG_CONNECTION_STRING"]).get_engine()

In [ ]:
import pkgutil

import assistant_agent.agents as agents_pkg

AGENT_IDS = [name for _, name, is_pkg in pkgutil.iter_modules(agents_pkg.__path__) if not is_pkg]

## Checkpointer
旧 thread_id（`agent_id` ごとの世代交代で使われなくなった checkpoint）を手動で削除する。

### 削除対象を表示（確認用）

In [ ]:
import itertools

from sqlalchemy import func, select

from assistant_agent.entities.checkpoint import CheckpointEntity

KEEP_LATEST_N = 0  # agent_id ごとに残す世代数

agent_id_col = func.split_part(CheckpointEntity.thread_id, ":", 1)
async with engine.connect() as conn:
    stmt = (
        select(
            agent_id_col.label("agent_id"),
            CheckpointEntity.thread_id,
            func.max(CheckpointEntity.checkpoint_id).label("latest"),
        )
        .group_by(agent_id_col, CheckpointEntity.thread_id)
        .order_by(agent_id_col, func.max(CheckpointEntity.checkpoint_id).desc())
    )
    rows = (await conn.execute(stmt)).all()

stale_thread_ids = []
for agent_id, group in itertools.groupby(rows, key=lambda row: row.agent_id):
    stale_thread_ids += [row.thread_id for row in list(group)[KEEP_LATEST_N:]]
print(f"Deleting {len(stale_thread_ids)} thread(s):\n{'\n'.join(stale_thread_ids)}")


### 実際に削除する（セル1を確認してから実行）

In [ ]:
from sqlalchemy import delete

from assistant_agent.entities.checkpoint import (
    CheckpointBlobEntity,
    CheckpointEntity,
    CheckpointWriteEntity,
)

async with engine.begin() as conn:
    for entity in (CheckpointEntity, CheckpointBlobEntity, CheckpointWriteEntity):
        await conn.execute(delete(entity).where(entity.thread_id.in_(stale_thread_ids)))


# Dispatcher (個別編集)

### レコードを表示

In [ ]:
import pandas as pd
from sqlalchemy import select

from assistant_agent.entities.postgres import DispatchEntity
from assistant_agent.store import PostgresStoreConnector

pg_conn = PostgresStoreConnector(os.environ["AA_PG_CONNECTION_STRING"])
stmt = select(DispatchEntity).order_by(DispatchEntity.run_at)
dispatch_df = pd.read_sql(stmt, pg_conn.get_engine_sync())
dispatch_df


### Dispatch 操作関数
以下の UI セルから呼び出す、DB 操作のみを行う関数（表示・入力処理は含まない）。

In [ ]:
import uuid
from datetime import datetime
from zoneinfo import ZoneInfo

from sqlalchemy import delete, insert, update

from assistant_agent.entities.postgres import DispatchEntity

ONE_SHOT = -1  # DispatcherService.ONE_SHOT と同じ値（単発予定）
JST = ZoneInfo("Asia/Tokyo")


async def insert_dispatch(
    agent_id: str, prompt: str, run_at: datetime, interval_seconds: int
) -> dict:
    """新しい予定を1件登録し、登録した内容を返す."""
    new_dispatch = {
        "dispatch_id": str(uuid.uuid7()),
        "agent_id": agent_id,
        "prompt": prompt,
        "interval_seconds": interval_seconds,
        "run_at": run_at,
    }
    async with engine.begin() as conn:
        await conn.execute(insert(DispatchEntity), [new_dispatch])
    return new_dispatch


async def update_dispatch(
    dispatch_id: str, prompt: str, run_at: datetime, interval_seconds: int
) -> int:
    """dispatch_id を指定して予定を更新し、更新件数を返す."""
    values = {"prompt": prompt, "interval_seconds": interval_seconds, "run_at": run_at}
    async with engine.begin() as conn:
        result = await conn.execute(
            update(DispatchEntity).where(DispatchEntity.dispatch_id == dispatch_id).values(**values)
        )
        return result.rowcount


async def delete_dispatch(dispatch_id: str) -> int:
    """dispatch_id を指定して予定を削除し、削除件数を返す."""
    async with engine.begin() as conn:
        result = await conn.execute(
            delete(DispatchEntity).where(DispatchEntity.dispatch_id == dispatch_id)
        )
        return result.rowcount

### 予定を追加

In [ ]:
import asyncio
from datetime import timedelta

import ipywidgets as widgets
from IPython.display import display

w_agent_id = widgets.Dropdown(options=AGENT_IDS, description="agent_id")
w_prompt = widgets.Textarea(
    placeholder="メンテナンス確認用の予定", description="prompt", layout=widgets.Layout(width="80%")
)
w_run_at = widgets.DatetimePicker(
    value=datetime.now(JST) + timedelta(seconds=10), description="run_at (JST)"
)
w_interval_seconds = widgets.IntText(value=ONE_SHOT, description="interval_seconds")
w_submit = widgets.Button(description="追加を実行", button_style="primary")
w_output = widgets.Output()


@w_output.capture(clear_output=True)
def _on_submit(_button: widgets.Button) -> None:
    async def _run() -> None:
        new_dispatch = await insert_dispatch(
            str(w_agent_id.value),
            str(w_prompt.value),
            w_run_at.value.replace(tzinfo=JST),
            w_interval_seconds.value,
        )
        print(new_dispatch)

    asyncio.create_task(_run())


w_submit.on_click(_on_submit)
display(w_agent_id, w_prompt, w_run_at, w_interval_seconds, w_submit, w_output)

### 予定を更新

In [ ]:
import ipywidgets as widgets
from IPython.display import display

w_dispatch_id = widgets.Dropdown(
    options=list(dispatch_df["dispatch_id"]) if not dispatch_df.empty else [],
    description="dispatch_id",
)
w_prompt = widgets.Textarea(description="prompt", layout=widgets.Layout(width="80%"))
w_run_at = widgets.DatetimePicker(description="run_at (JST)")
w_interval_seconds = widgets.IntText(description="interval_seconds")
w_submit = widgets.Button(description="更新を実行", button_style="primary")
w_output = widgets.Output()


def _on_select(change: dict) -> None:
    dispatch_id = change["new"]
    if not dispatch_id:
        return
    row = dispatch_df.loc[dispatch_df["dispatch_id"] == dispatch_id].iloc[0]
    w_prompt.value = row["prompt"]
    w_run_at.value = row["run_at"].astimezone(JST)
    w_interval_seconds.value = row["interval_seconds"]


@w_output.capture(clear_output=True)
def _on_submit(_button: widgets.Button) -> None:
    dispatch_id = w_dispatch_id.value
    assert dispatch_id is not None

    async def _run() -> None:
        rowcount = await update_dispatch(
            dispatch_id,
            w_prompt.value,
            w_run_at.value.replace(tzinfo=JST),
            w_interval_seconds.value,
        )
        print(f"Updated {rowcount} row(s): {dispatch_id}")

    asyncio.create_task(_run())


w_dispatch_id.observe(_on_select, names="value")
w_submit.on_click(_on_submit)
if w_dispatch_id.options:
    _on_select({"new": w_dispatch_id.value})
display(w_dispatch_id, w_prompt, w_run_at, w_interval_seconds, w_submit, w_output)

### 予定を削除

In [ ]:
import ipywidgets as widgets
from IPython.display import display

w_dispatch_id = widgets.Dropdown(
    options=list(dispatch_df["dispatch_id"]) if not dispatch_df.empty else [],
    description="dispatch_id",
)
w_preview = widgets.HTML()
w_submit = widgets.Button(description="削除を実行", button_style="danger")
w_output = widgets.Output()


def _on_select(change: dict) -> None:
    dispatch_id = change["new"]
    if not dispatch_id:
        w_preview.value = ""
        return
    row = dispatch_df.loc[dispatch_df["dispatch_id"] == dispatch_id].iloc[0]
    w_preview.value = f"<b>run_at:</b> {row['run_at']}<br><b>prompt:</b><br>{row['prompt'][:200]}"


@w_output.capture(clear_output=True)
def _on_submit(_button: widgets.Button) -> None:
    dispatch_id = w_dispatch_id.value
    assert dispatch_id is not None

    async def _run() -> None:
        rowcount = await delete_dispatch(dispatch_id)
        print(f"Deleted {rowcount} row(s): {dispatch_id}")

    asyncio.create_task(_run())


w_dispatch_id.observe(_on_select, names="value")
w_submit.on_click(_on_submit)
if w_dispatch_id.options:
    _on_select({"new": w_dispatch_id.value})
display(w_dispatch_id, w_preview, w_submit, w_output)

# Dispatcher (一括保守)
予約を Markdown ファイルへ書き出し・取り込みする。実行前に対象 `agent_id` の AgentBot を停止すること（動作中の取り込み結果は保証しない）。

In [ ]:
from pathlib import Path

from assistant_agent.entities.postgres import DispatchEntity
from assistant_agent.utils.dispatcher import DispatcherMaintenance

maintenance = DispatcherMaintenance(
    dispatch_entity=DispatchEntity, sa_engine=engine, root=Path("../docs/dispatches")
)


### 予約を書き出し
対象 AgentBot を停止してから実行すること。

In [ ]:
agent_id = "holo"  # AGENT_IDS から対象を選ぶ

paths = await maintenance.dump(agent_id)
print(f"Dumped {len(paths)} file(s):")
for path in paths:
    print(f"  {path}")

### 予約を取り込む
対象 AgentBot を停止してから実行すること。指定 `agent_id` の既存予約をファイル集合で洗い替える不可逆操作。AgentBot 動作中の取り込み結果は保証しない。

In [ ]:
agent_id = "holo"  # AGENT_IDS から対象を選ぶ

dispatches = await maintenance.load(agent_id)
print(f"Loaded {len(dispatches)} dispatch(es)")

# Skills
`skills_metadata`（SKILL.md frontmatter 由来）は最初の1回だけ checkpoint state へ読み込まれ、以降のセッションでは再読込されない
（[SkillsMiddleware.before_agent](../.venv/lib/python3.14/site-packages/deepagents/middleware/skills.py)）。

SKILL.md の name/description 変更やスキルの追加・削除を、checkpoint を消さずに反映するための暫定処置として、
deepagents のプライベート関数 `_list_skills_with_errors` を直接呼んで最新のメタデータを計算し、対象 thread の state を `aupdate_state()` で上書きする。
SKILL.md **本文**のみの変更（frontmatter 以外）は `read_file` で都度読まれるため、この節の操作は不要。

deepagents の非公開実装（`_list_skills_with_errors`）に依存しているため、deepagents のバージョンアップで動作しなくなる可能性がある。

### 対象 agent 用のグラフを構築（推論は行わない。state 更新のみに使う）

In [14]:
import os

from deepagents.middleware.skills import SkillsMiddleware
from langchain.agents import create_agent
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver

from assistant_agent import agents
from assistant_agent.agent_bot import find_latest_thread_id
from assistant_agent.store import PostgresStoreConnector

skills_agent_id = "sample"  # AGENT_IDS から対象を選ぶ
skills_thread_id = await find_latest_thread_id(engine, skills_agent_id)
assert skills_thread_id is not None, f"no thread found for agent_id={skills_agent_id!r}"
skills_backend = agents.build_backend(skills_agent_id)

# 後続セルをまたいで使うため、pool は async with を使わずここで開き、
# 最後のセルで明示的に close する（開いたまま notebook を終了しないこと）。
skills_pg_conn = PostgresStoreConnector(os.environ["AA_PG_CONNECTION_STRING"])
skills_pool = skills_pg_conn.get_psycopg_pool()
await skills_pool.open()
skills_checkpointer = AsyncPostgresSaver(conn=skills_pool)
await skills_checkpointer.setup()

# 推論は一切行わないダミーモデル。エージェント本体の初期化（ツール一式・LLM接続・
# context構築）を避け、SkillsMiddleware だけを積んだ最小グラフで aupdate_state を呼ぶ。
skills_graph = create_agent(
    FakeListChatModel(responses=[]),
    middleware=[SkillsMiddleware(backend=skills_backend, sources=["/skills"])],
    checkpointer=skills_checkpointer,
)

print(f"thread_id: {skills_thread_id}")


thread_id: sample:01a0043d-50dc-706d-802e-17a663232e5f


### skills_metadata を再計算して確認

In [15]:
from deepagents.middleware.skills import _list_skills_with_errors

new_skills_metadata: list = []
new_skills_load_errors: list = []
for source_path in ["/skills"]:  # 対象 agent の build_agent() に渡している skills= と合わせる
    source_skills, source_error = _list_skills_with_errors(skills_backend, source_path)
    if source_error is not None:
        new_skills_load_errors.append(source_error)
    new_skills_metadata.extend(source_skills)

print(f"Loaded {len(new_skills_metadata)} skill(s): {[s['name'] for s in new_skills_metadata]}")
if new_skills_load_errors:
    print(f"Load errors: {new_skills_load_errors}")


Loaded 6 skill(s): ['arxiv-search', 'file-organizer', 'langgraph-docs', 'news-watch', 'skill-creator', 'web-research']


### 対象 thread の state を上書き（内容を確認してから実行）

In [ ]:
from langchain_core.runnables import RunnableConfig

skills_config: RunnableConfig = {"configurable": {"thread_id": skills_thread_id}}
# as_node を省略すると複数ノード時に更新先を特定できず、例外を出さないまま
# 書き込みが無視される（state に反映されない）ため、ノード名を明示する。
await skills_graph.aupdate_state(
    skills_config,
    {"skills_metadata": new_skills_metadata, "skills_load_errors": new_skills_load_errors},
    as_node="SkillsMiddleware.before_agent",
)

snapshot = await skills_graph.aget_state(skills_config)
assert snapshot.values.get("skills_metadata") == new_skills_metadata, (
    "state was not updated as expected"
)
print("Updated skills_metadata in checkpoint state.")

Updated skills_metadata in checkpoint state.


In [17]:
await skills_pool.close()